In [ ]:
# ==========================================================
# ULTRA-FAST HIGH-ACCURACY LIGHTGBM (< 3 MINUTE RUNTIME)
# train.csv + test.csv + sample_submission.csv
# ==========================================================

# ============================
# 1. Import Libraries
# ============================
import pandas as pd
import numpy as np
from pathlib import Path
from lightgbm import LGBMClassifier, early_stopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# ============================
# 2. Load Data
# ============================

KAGGLE_DATASET_URL = "/kaggle/input/competitions/playground-series-s6e8/"


def find_data_dir() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "notebook",
        cwd.parent,
        cwd.parent / "notebook",
    ])

    for base in candidates:
        for candidate in [base / "input", base.parent / "input"]:
            if candidate.exists() and (candidate / "train.csv").exists() and (candidate / "test.csv").exists():
                return candidate

    for kaggle_path in [
        Path(KAGGLE_DATASET_URL),
        Path("/kaggle/input/train.csv").parent,
    ]:
        if kaggle_path.exists() and (kaggle_path / "train.csv").exists() and (kaggle_path / "test.csv").exists():
            return kaggle_path

    return Path(KAGGLE_DATASET_URL)


data_dir = find_data_dir()

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

# ============================
# 3. Separate X and y
# ============================
TARGET = "addicted_label"
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_test = test.copy()


# ============================
# 4. Fast Categorical Conversion
# ============================
# LightGBM handles text columns natively if cast to 'category'.
# This bypasses pd.get_dummies entirely, slashing runtime drastically.
object_cols = X.select_dtypes(include=["object", "string"]).columns

for col in object_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")


# ============================
# 5. Encode Target
# ============================
encoder = LabelEncoder()
y = encoder.fit_transform(y)


# ============================
# 6. Validation Split
# ============================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# ============================
# 7. Fast-Converging LightGBM Model
# ============================
# Higher learning rate (0.05) speeds up convergence significantly.
# Pre-set regularizations maintain high accuracy at high speeds.
model = LGBMClassifier(  
    n_estimators=514,#550,
    learning_rate=0.05,
    num_leaves=25,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


# ============================
# 8. Train with Early Stopping
# ============================
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(stopping_rounds=30, verbose=False)]
)


# ============================
# 9. Validation Score
# ============================
val_pred = model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_pred))
# Validation Accuracy: 0.8955190419023099

# ============================
# 10. Instant Full Data Retrain
# ============================
# Pull the exact number of trees needed to avoid overfitting and save time.
best_iteration = model.best_iteration_ if model.best_iteration_ else 150

final_model = LGBMClassifier(  
    n_estimators=best_iteration,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
final_model.fit(X, y)


# ============================
# 11. Predict Test
# ============================
prediction = final_model.predict(X_test)
prediction = encoder.inverse_transform(prediction)


# ============================
# 12. Submission
# ============================
sample_submission[TARGET] = prediction
sample_submission.to_parquet("../output/submission.parquet", index=False)

print("submission.parquet created successfully!")

Validation Accuracy: 0.8955190419023099
submission.parquet created successfully!
